In [7]:
import pandas as pd
import numpy as np

In [20]:
!pip install shutil

ERROR: Could not find a version that satisfies the requirement shutil (from versions: none)
ERROR: No matching distribution found for shutil


In [19]:
import shutil

shutil.copytree("/workspace/SentiChat/ml_service/dataset", "/content/dataset")

FileNotFoundError: [Errno 2] No such file or directory: '/workspace/SentiChat/ml_service/dataset'

In [12]:
import os
print(os.getcwd())

/content


In [13]:
df1 = pd.read_csv('/content/dataset/goemotions_1.csv')
# df2 = pd.read_csv('dataset/goemotions_2.csv')
# df3 = pd.read_csv('dataset/goemotions_3.csv')

# df = pd.concat([df1, df2, df3], ignore_index=True)

FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset/goemotions_1.csv'

In [6]:
df.head()

,text,id,author,subreddit,link_id,parent_id,created_utc,rater_id,example_very_unclear,admiration,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,That game hurt.,eew5j0j,Brdd9,nrl,t3_ajis4z,t1_eew18eq,1.548381e+09,1,False,0,...,0,0,0,0,0,0,0,1,0,0
1,>sexuality shouldn’t be a grouping category I...,eemcysk,TheGreen888,unpopularopinion,t3_ai4q37,t3_ai4q37,1.548084e+09,37,True,0,...,0,0,0,0,0,0,0,0,0,0
2,"You do right, if you don't care then fuck 'em!",ed2mah1,Labalool,confessions,t3_abru74,t1_ed2m7g7,1.546428e+09,37,False,0,...,0,0,0,0,0,0,0,0,0,1
3,Man I love reddit.,eeibobj,MrsRobertshaw,facepalm,t3_ahulml,t3_ahulml,1.547965e+09,18,False,0,...,1,0,0,0,0,0,0,0,0,0
4,"[NAME] was nowhere near them, he was by the Fa...",eda6yn6,American_Fascist713,starwarsspeculation,t3_ackt2f,t1_eda65q2,1.546669e+09,2,False,0,...,0,0,0,0,0,0,0,0,0,1


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211225 entries, 0 to 211224
Data columns (total 37 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   text                  211225 non-null  object 
 1   id                    211225 non-null  object 
 2   author                211225 non-null  object 
 3   subreddit             211225 non-null  object 
 4   link_id               211225 non-null  object 
 5   parent_id             211225 non-null  object 
 6   created_utc           211225 non-null  float64
 7   rater_id              211225 non-null  int64  
 8   example_very_unclear  211225 non-null  bool   
 9   admiration            211225 non-null  int64  
 10  amusement             211225 non-null  int64  
 11  anger                 211225 non-null  int64  
 12  annoyance             211225 non-null  int64  
 13  approval              211225 non-null  int64  
 14  caring                211225 non-null  int64  
 15  

In [8]:
df.shape

(211225, 37)

In [9]:
emotion_cols = df.columns[9:]
emotion_cols = df.columns.difference([
    "text","id","author","subreddit","link_id",
    "parent_id","created_utc","rater_id","example_very_unclear"
])

In [10]:
df["label"] = df[emotion_cols].idxmax(axis=1)

In [11]:
X = df["text"]
y = df["label"]

In [12]:
df = df[["text","label"]]
df = df.reset_index(drop=True)

In [13]:
df.head()

,text,label
0,That game hurt.,sadness
1,>sexuality shouldn’t be a grouping category I...,admiration
2,"You do right, if you don't care then fuck 'em!",neutral
3,Man I love reddit.,love
4,"[NAME] was nowhere near them, he was by the Fa...",neutral


In [14]:
positive_emotions = [
    "admiration","amusement","approval","caring","desire",
    "excitement","gratitude","joy","love","optimism",
    "pride","relief"
]

negative_emotions = [
    "anger","annoyance","disappointment","disapproval",
    "disgust","embarrassment","fear","grief","nervousness",
    "remorse","sadness"
]

def convert_sentiment(label):
    if label in positive_emotions:
        return "positive"
    elif label in negative_emotions:
        return "negative"
    else:
        return "neutral"

df["label"] = df["label"].apply(convert_sentiment)

In [15]:
df.sample(10)

,text,label
184282,That is odd.,negative
115560,Yeah that is a pretty crap and unhelpful thing...,negative
30497,This is surprising as many pizza places us pea...,neutral
66425,That’s a lot of chonkers!,positive
142355,"Agreed, but I can't believe the bottom of the ...",positive
37359,Only married once so can't be that bad. Not th...,negative
35073,"Lived this, it didn’t workout for me. Good luc...",positive
194058,Audio version as well.,positive
120492,I don’t know where you live but just a bj shou...,neutral
167716,“Siri. Call Delivery.”,positive


In [16]:
df['label'].value_counts()

,count
label,
positive,82847
neutral,78202
negative,50176


In [17]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<user>', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'>.*\n?', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df["text"] = df["text"].apply(clean_text)

In [18]:
from sklearn.model_selection import train_test_split

X = df["text"]
y = df["label"]
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42,test_size=0.2,stratify=y)

In [19]:
from sklearn.preprocessing import LabelEncoder
ln =  LabelEncoder()
y_train = ln.fit_transform(y_train)
y_test  = ln.transform(y_test)

In [20]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 30000
max_len = 50

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding="post")

In [21]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))


In [22]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [23]:
from keras import Sequential
from keras.layers import Dense,Flatten,Dropout,Embedding,Bidirectional,Input,LSTM

In [24]:
model = Sequential()
model.add(Input(shape=(max_len,)))
model.add(Embedding(vocab_size,128))
model.add(Bidirectional(LSTM(256)))
model.add(Dropout(0.2))
model.add(Dense(64,activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(5,activation='softmax'))
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 50, 128)        │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 512)            │       788,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        32,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,661,637 (17.78 MB)

 Trainable params: 4,661,637 (17.78 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

In [ ]:
history = model.fit(X_train_pad,y_train,epochs=15,batch_size=64,validation_data=(X_test_pad,y_test),class_weight=class_weights,callbacks=[early_stop])

Epoch 1/15
2641/2641 ━━━━━━━━━━━━━━━━━━━━ 48s 18ms/step - accuracy: 0.5975 - loss: 0.8731 - val_accuracy: 0.6259 - val_loss: 0.8256
Epoch 2/15
2641/2641 ━━━━━━━━━━━━━━━━━━━━ 44s 16ms/step - accuracy: 0.6571 - loss: 0.7676 - val_accuracy: 0.6325 - val_loss: 0.8104
Epoch 3/15
2641/2641 ━━━━━━━━━━━━━━━━━━━━ 44s 17ms/step - accuracy: 0.6797 - loss: 0.7138 - val_accuracy: 0.6420 - val_loss: 0.8240
Epoch 4/15
2641/2641 ━━━━━━━━━━━━━━━━━━━━ 44s 17ms/step - accuracy: 0.6912 - loss: 0.6723 - val_accuracy: 0.6389 - val_loss: 0.8493
Epoch 5/15
2641/2641 ━━━━━━━━━━━━━━━━━━━━ 43s 16ms/step - accuracy: 0.7008 - loss: 0.6388 - val_accuracy: 0.6356 - val_loss: 0.8992


# Using Word2Vec embeddings

In [25]:
embedding_dim = 200
embedding_index = {}
with open("glove.6B.200d.txt", encoding="utf8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        embedding_index[word] = vector

In [26]:
import numpy as np
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in tokenizer.word_index.items():
    if i >= vocab_size:
        continue
    embedding_vector = embedding_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

In [27]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional, LSTM,
    Dense, Dropout, Attention, GlobalAveragePooling1D
)

inputs = Input(shape=(max_len,))

x = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    weights=[embedding_matrix],
    trainable=True
)(inputs)

x = Bidirectional(LSTM(128, return_sequences=True))(x)

# Attention
x = Attention()([x, x])

# Collapse sequence dimension
x = GlobalAveragePooling1D()(x)

x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)

outputs = Dense(3, activation='softmax')(x)

model = Model(inputs, outputs)

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 50, 200)   │  6,000,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 50, 256)   │    336,896 │ embedding_1[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 50, 256)   │          0 │ bidirectional_1[… │
│ (Attention)         │                   │            │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ attention[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │     16,448 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 3)         │        195 │ dropout_2[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,353,539 (24.24 MB)

 Trainable params: 6,353,539 (24.24 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_test_pad, y_test),
    epochs=15,
    batch_size=64,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Epoch 1/15
2641/2641 ━━━━━━━━━━━━━━━━━━━━ 54s 18ms/step - accuracy: 0.6080 - loss: 0.8513 - val_accuracy: 0.6353 - val_loss: 0.8004
Epoch 2/15
2641/2641 ━━━━━━━━━━━━━━━━━━━━ 45s 17ms/step - accuracy: 0.6579 - loss: 0.7618 - val_accuracy: 0.6359 - val_loss: 0.8013
Epoch 3/15
2641/2641 ━━━━━━━━━━━━━━━━━━━━ 44s 17ms/step - accuracy: 0.6819 - loss: 0.7079 - val_accuracy: 0.6437 - val_loss: 0.8028
Epoch 4/15
2641/2641 ━━━━━━━━━━━━━━━━━━━━ 43s 16ms/step - accuracy: 0.6953 - loss: 0.6637 - val_accuracy: 0.6384 - val_loss: 0.8473


# Prediction

In [38]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
label_map={
    0: "negative",
    1: "neutral",
    2: "positive"
}

In [31]:
def predict(sentence):
  sequence = tokenizer.texts_to_sequences([sentences])
  padded = pad_sequences(sequence,maxlen=max_len,padding='post')
  pred = model.predict(padded)

  class_id = np.argmax(pred)
  confidence = float(np.max(pred))

  return {
      'sentence':sentence,
      'label':label_map[class_id],
      'confidence':confidence
  }


In [42]:
sentences = 'I would definitely recommend this to everyone.'
predict(sentences)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step


{'sentence': 'I would definitely recommend this to everyone.',
 'label': 'positive',
 'confidence': 0.6187836527824402}

In [44]:
sentences = 'The update completely ruined the app.'
predict(sentences)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step


{'sentence': 'The update completely ruined the app.',
 'label': 'negative',
 'confidence': 0.6933231353759766}

In [43]:
sentences = 'This is a standard version of the software.'
predict(sentences)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step


{'sentence': 'This is a standard version of the software.',
 'label': 'neutral',
 'confidence': 0.7432668805122375}

# model problem - sarcasm

In [55]:
sentences = 'Great, the app crashed again.'
predict(sentences)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step


{'sentence': 'Great, the app crashed again.',
 'label': 'positive',
 'confidence': 0.8825671076774597}

#Pickeling

In [ ]:
model.save("sentiment_lstm.keras")

In [59]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

with open("max_len.pkl", "wb") as f:
    pickle.dump(max_len, f)